# FM — Factorization Machines (2010)

_Papers · Recommender Systems_

**Learn every pairwise feature interaction with a shared low-rank vector per feature, computed in linear time.**

---

This notebook is a paced, step-by-step walkthrough of the **FM — Factorization Machines (2010)** lesson. We build the model one piece at a time: the O(kn) interaction trick by hand, a batched FM, and a head-to-head against plain linear regression. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build FM in four steps: (1) verify the O(kn) interaction trick by hand on a tiny case, (2) write a batched FM and check it against the slow O(kn²) oracle, (3) build interaction-only data and the two models, (4) fit both and compare.

### Step 1 — Check the O(kn) trick by hand

The 2-way FM interaction is a double sum over all feature pairs $i<j$ of $\langle v_i, v_j\rangle x_i x_j$ — that costs $O(kn^2)$. Lemma 3.1 rewrites it as $\tfrac12\sum_f\big[(\sum_i v_{if}x_i)^2 - \sum_i v_{if}^2 x_i^2\big]$, which is only $O(kn)$.

Here we compute both forms on a tiny case ($n=3$, $k=2$) and confirm they give the same number. The squared-sum carries the cross terms we want; subtracting the sum-of-squares removes the diagonal, and halving leaves exactly the pairs.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Worked example by hand: x = [1, 2, 1], k = 2 factors.
xw = torch.tensor([1.0, 2.0, 1.0])              # n = 3 feature values
Vw = torch.tensor([[1.0, 1.0],                  # v_1
                   [2.0, 0.0],                  # v_2
                   [0.0, 1.0]])                 # v_3   (shape n x k = 3 x 2)

# Naive double sum over pairs i < j -- the O(k n^2) definition.
naive = 0.0
for i in range(3):
    for j in range(i + 1, 3):
        naive += (Vw[i] @ Vw[j]) * xw[i] * xw[j]

# O(kn) form: 0.5 * sum_f [ (sum_i v_if x_i)^2 - sum_i v_if^2 x_i^2 ]
sq_of_sum = (xw @ Vw) ** 2                       # per factor: (sum_i v_if x_i)^2
sum_of_sq = (xw ** 2) @ (Vw ** 2)                # per factor: sum_i v_if^2 x_i^2
fast = 0.5 * (sq_of_sum - sum_of_sq).sum()

print("worked example  naive =", float(naive), " fast =", fast.item())   # both 5.0

### Step 2 — A batched FM, checked against the slow oracle

Now we wrap the O(kn) form into `fm_fast`, which scores a whole batch at once: a linear part $w_0 + x\cdot w$ plus the fast interaction term. To trust it, we also write `fm_naive`, the literal $O(kn^2)$ double-sum oracle.

We feed both the same random weights and inputs and check they agree to floating-point tolerance. Matching here means our fast form is the same function, just faster.

In [ ]:
def fm_fast(x, w0, w, V):
    lin = w0 + x @ w
    sq_of_sum = (x @ V) ** 2                      # (B, k)
    sum_of_sq = (x ** 2) @ (V ** 2)               # (B, k)
    inter = 0.5 * (sq_of_sum - sum_of_sq).sum(dim=1)
    return lin + inter

def fm_naive(x, w0, w, V):                        # O(k n^2) reference -- the oracle
    lin = x @ w + w0
    B, n = x.shape
    inter = torch.zeros(B)
    for i in range(n):
        for j in range(i + 1, n):
            inter += (V[i] @ V[j]) * x[:, i] * x[:, j]
    return lin + inter

n, k, B = 6, 4, 5
w0 = torch.randn(())
w  = torch.randn(n)
V  = torch.randn(n, k)
x  = torch.randn(B, n)

yf = fm_fast(x, w0, w, V)
yn = fm_naive(x, w0, w, V)
print("fast == naive (allclose):", torch.allclose(yf, yn, atol=1e-5))   # True
print("max abs diff:", (yf - yn).abs().max().item())                    # ~1e-6

### Step 3 — Build interaction-only data and the two models

To show *why* FM matters, we generate data whose target lives **entirely** in feature products: $y = 1.5\,x_0 x_1 + 1.0\,x_2 x_3 + \text{noise}$, with $\pm 1$ features. There is no useful linear signal — averaged over $\pm 1$ inputs, each single feature is uninformative.

We define two models: an `FM` (linear part plus the factorized interaction term) and a `LinearReg` ablation (the same model with the interaction term removed). They differ in exactly one thing, so any gap is attributable to the pairwise term.

In [ ]:
# Reproduce the effect: FM should beat plain linear regression on interaction data.
N, n = 600, 8
g = torch.Generator().manual_seed(1)
X = (torch.rand(N, n, generator=g) < 0.5).float() * 2 - 1               # +/-1 features

# Target lives ENTIRELY in feature products -> no useful linear signal.
y = 1.5 * X[:, 0] * X[:, 1] + 1.0 * X[:, 2] * X[:, 3] + 0.2 * torch.randn(N, generator=g)

ntr = 480
Xtr, ytr = X[:ntr], y[:ntr]
Xte, yte = X[ntr:], y[ntr:]

class FM(nn.Module):
    def __init__(self, n, k):
        super().__init__()
        self.w0 = nn.Parameter(torch.zeros(()))
        self.w  = nn.Parameter(torch.zeros(n))
        self.V  = nn.Parameter(torch.randn(n, k) * 0.1)
    def forward(self, x):
        return fm_fast(x, self.w0, self.w, self.V)

class LinearReg(nn.Module):                       # the ablation: FM with the interaction term removed
    def __init__(self, n):
        super().__init__()
        self.w0 = nn.Parameter(torch.zeros(()))
        self.w  = nn.Parameter(torch.zeros(n))
    def forward(self, x):
        return self.w0 + x @ self.w

### Step 4 — Fit both and compare test error

We fit each model with Adam on mean-squared error, then read the test MSE. We reseed before each fit so the only difference between the two runs is the model class.

The plain linear model should land near the *variance* of the target — it can only predict the mean — while the FM should be far lower. That gap pins the win on the factorized interaction term, exactly the paper's point.

In [ ]:
def fit(model, epochs=300, lr=0.05):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lf = nn.MSELoss()
    for _ in range(epochs):
        opt.zero_grad()
        loss = lf(model(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        return lf(model(Xte), yte).item()

torch.manual_seed(0)
fm_mse = fit(FM(n, k=5))

torch.manual_seed(0)
lin_mse = fit(LinearReg(n))

print("var(y_test)        =", round(yte.var(unbiased=False).item(), 4))   # ~2.97
print("Linear  test MSE   =", round(lin_mse, 4))                          # ~2.91 (learns ~nothing)
print("FM (k=5) test MSE  =", round(fm_mse, 4))                           # ~0.047
# Our small run, not the paper's number: FM crushes linear when the signal is in feature products.

## Visualize the data & results

_On data whose target depends only on feature products, does the FM's interaction term beat plain linear regression as training proceeds?_

### Step 1 — Rebuild the data and models for a learning-curve run

This panel re-runs the FM-vs-linear comparison but records the test error after **every** epoch so we can see the curves diverge over time. We rebuild the same interaction-only data and the same two model classes here so the cell is self-contained.

The `FM` here writes the interaction term inline (the same O(kn) Lemma 3.1 form as before); `Lin` is the interaction-free ablation.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# FM (with interaction term) vs plain linear regression on interaction-only data.
N, n = 600, 8
g = torch.Generator().manual_seed(1)
X = (torch.rand(N, n, generator=g) < 0.5).float() * 2 - 1
y = 1.5 * X[:, 0] * X[:, 1] + 1.0 * X[:, 2] * X[:, 3] + 0.2 * torch.randn(N, generator=g)

ntr = 480
Xtr, ytr = X[:ntr], y[:ntr]
Xte, yte = X[ntr:], y[ntr:]

class FM(nn.Module):
    def __init__(self, n, k):
        super().__init__()
        self.w0 = nn.Parameter(torch.zeros(()))
        self.w  = nn.Parameter(torch.zeros(n))
        self.V  = nn.Parameter(torch.randn(n, k) * 0.1)
    def forward(self, x):                          # O(kn) interaction term (Lemma 3.1)
        sq_of_sum = (x @ self.V) ** 2
        sum_of_sq = (x ** 2) @ (self.V ** 2)
        inter = 0.5 * (sq_of_sum - sum_of_sq).sum(1)
        return self.w0 + x @ self.w + inter

class Lin(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.w0 = nn.Parameter(torch.zeros(()))
        self.w  = nn.Parameter(torch.zeros(n))
    def forward(self, x):
        return self.w0 + x @ self.w

### Step 2 — Record the test-error curve for each model

`curve` fits a model and appends the test MSE after every epoch, returning the whole trajectory. We run it once for the FM and once for the linear model, reseeding so the only difference is the model.

We then sample 30 evenly-spaced points along each curve and print them, so you can watch the FM's error plunge while the linear model stays stuck near the target's variance.

In [ ]:
def curve(model, epochs=300, lr=0.05):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lf = nn.MSELoss()
    te = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = lf(model(Xtr), ytr)
        loss.backward()
        opt.step()
        with torch.no_grad():
            te.append(lf(model(Xte), yte).item())
    return te

torch.manual_seed(0)
fm_c = curve(FM(n, 5))

torch.manual_seed(0)
lin_c = curve(Lin(n))

idx = np.linspace(0, 299, 30).astype(int)
print("FM :", [[int(i), round(fm_c[i], 4)]  for i in idx])
print("LIN:", [[int(i), round(lin_c[i], 4)] for i in idx])
# FM -> ~0.047 test MSE; Linear stuck at ~2.91 (~variance of y). Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You built data whose target depends only on feature products (e.g.
            $x_0 x_1$ and $x_2 x_3$) and fit an FM that reaches low test error. Now remove the interaction term
            &mdash; keep only $w_0 + \sum_i w_i x_i$ (a plain linear model of the same features) &mdash; and
            refit. What happens to test error, and what does that isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete the interaction term so the prediction is just $w_0 + x \cdot w$; keep the same features, optimizer, and data. — _An honest ablation changes exactly one thing &mdash; the $\langle v_i,v_j\rangle x_i x_j$ term &mdash; so any difference is attributable to it._
- Refit and read the test mean-squared error: the linear model stays near the variance of the target (it cannot fit the products), while the FM was far lower. — _A single weight per feature cannot represent $x_i x_j$; with $\pm 1$ features the product averages out and the linear part learns nothing useful._
- Conclude that the factorized interaction term, not extra features or capacity, is what fits the signal. — _Both models see identical features; only the FM has the pairwise term, so the gap pins the win on it._

**Answer:** The plain linear model's test error sits near the target's variance &mdash; it essentially
                 predicts the mean &mdash; while the FM's is far lower. Since the two models differ only in the
                 $\langle v_i,v_j\rangle x_i x_j$ term, this isolates the factorized interaction as the
                 reason FM fits interaction-driven data. The CODEVIZ panel shows exactly this contrast.

</details>

**Problem 2.** Confirm the O(kn) trick by hand on a fresh tiny case. Take $n = 2$, $k = 1$, values
            $x = [\,3,\ 1\,]$, and one factor per feature $v_1 = [\,2\,]$, $v_2 = [\,1\,]$. Compute the
            interaction both ways and check they agree.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Naive: there is one pair $(1,2)$. $\langle v_1,v_2\rangle = 2\cdot 1 = 2$; times $x_1 x_2 = 3\cdot 1 = 3$ gives $2\cdot 3 = 6$. — _With two features there is exactly one unordered pair, so the double sum is a single term._
- Fast: $\sum_i v_{i,1} x_i = 2\cdot 3 + 1\cdot 1 = 7$, so the squared sum is $49$. Sum of squares $= 4\cdot 9 + 1\cdot 1 = 37$. — _One factor ($k=1$), so a single weighted sum, squared, minus the diagonal terms._
- Combine: $\tfrac{1}{2}(49 - 37) = \tfrac{1}{2}\cdot 12 = 6$. — _Square-of-sum minus sum-of-squares, halved &mdash; the lemma's right-hand side._

**Answer:** Both give 6. The fast form $\tfrac{1}{2}(49 - 37) = 6$ equals the single pair term
                 $\langle v_1,v_2\rangle x_1 x_2 = 6$, confirming Lemma 3.1 on this case &mdash; the squared
                 sum carries the cross term $2(v_{1}x_1)(v_{2}x_2)$ that we want, and subtracting the squares
                 then halving leaves exactly it.

</details>

**Problem 3.** A polynomial SVM gives the pair $(A, S)$ its own free weight $w_{A,S}$, while the FM uses
            $\langle v_A, v_S \rangle$. Suppose features $A$ and $S$ never co-occur in any training
            example. What happens to each model's estimate of that pair's interaction, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For the SVM: the gradient of the loss with respect to $w_{A,S}$ is proportional to $x_A x_S$, which is zero in every training example where they do not co-occur. — _A free pair weight only receives a gradient when both features are non-zero together; with no such example it never updates._
- For the FM: $v_A$ gets gradients from every example where $A$ co-occurs with some other feature, and $v_S$ likewise from its own co-occurrences. — _Each factor vector is shared across all pairs the feature joins, so it is trained even when the specific pair $(A,S)$ is absent._
- The FM's estimate $\langle v_A, v_S \rangle$ is then well-defined from two trained vectors, despite the pair never being seen. — _The interaction weight is computed from shared parameters, not stored per pair &mdash; this is the paper's sparsity argument (&sect;III-A3)._

**Answer:** The SVM's $w_{A,S}$ stays at its initial value &mdash; it never receives a gradient because
                 $x_A x_S = 0$ in every example &mdash; so the interaction is effectively unlearnable. The FM's
                 $\langle v_A, v_S \rangle$ is built from $v_A$ and $v_S$, each trained on the pairs those
                 features do appear in, so the unseen pair still gets a meaningful weight. This shared,
                 factorized parameterization is exactly why FMs learn interactions under sparsity.

</details>